In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE mavenmarket.gold.fact_returns
CLUSTER BY (return_date_key, store_sk)
COMMENT 'Product Returns Fact Table'
AS
WITH parsed_returns AS (
    SELECT 
        -- Date parsing logic for robust format handling
        COALESCE(
            to_date(return_date, 'MM/dd/yyyy'),
            to_date(return_date, 'dd-MMM-yy'),
            to_date(return_date, 'yyyy-MM-dd'),
            TRY_CAST(return_date AS DATE)
        ) AS parsed_return_date,
        
        product_sk,
        store_sk,
        COALESCE(quantity, 0) AS return_quantity
    FROM STREAM(mavenmarket.silver.silver_returns)
    WHERE return_date IS NOT NULL
)
SELECT 
    -- Primary Surrogate Key (Deterministic hash based on return date and foreign keys)
    ABS(HASH(CONCAT(
        COALESCE(CAST(r.parsed_return_date AS STRING), ''),
        COALESCE(r.product_sk, ''),
        COALESCE(r.store_sk, '')
    ))) AS returns_fact_sk,
    
    -- Date Foreign Key (mapped directly to dim_date.date_sk)
    COALESCE(
        d.date_sk, 
        CAST(date_format(r.parsed_return_date, 'yyyyMMdd') AS INT)
    ) AS return_date_key,
    
    -- Direct Surrogate Keys from Silver
    r.product_sk,
    r.store_sk,
    
    r.return_quantity

FROM parsed_returns r

-- Stream-to-Batch Join with Gold Date Dimension
LEFT JOIN mavenmarket.gold.dim_date d 
    ON r.parsed_return_date = d.calendar_date

WHERE r.parsed_return_date IS NOT NULL;